<a href="https://colab.research.google.com/github/malikasadnadir-max/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/malikasadnadir-max/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## 1. Two Paper Findings + My Methodology Questions

### Finding 1 — The Anatomy of Growing Content

The paper reports that growing pages tend to be longer, younger, and slightly better positioned than declining pages. In the reported comparison, growing pages averaged about 3.2K words and 184 days of age, while declining pages averaged about 2.3K words and 230 days of age. The paper explicitly describes this as an observational comparison rather than causal evidence.

**Where does the label/outcome come from?**

The paper defines trend direction from the change in impressions between the most recent 30-day period and the previous 30-day period. Pages are classified as Up when growth is greater than 10% and Down when decline is greater than 10%. Therefore, the outcome is derived from search-performance data rather than from a human quality judgment.

**My methodology question:**

Does the validation design adequately separate descriptive relationships from predictive evidence? In particular, if content age and word count are compared with a trend label derived from the same performance snapshot, a future or held-out time window would be needed before treating these variables as evidence of generalizable predictive performance.

The paper itself is careful to describe this finding as observational, which is appropriate. I would therefore treat it as a useful hypothesis for model development rather than proof that increasing word count or changing content age will cause growth.

### Finding 2 — The Content Performance Curve

The paper reports a lifecycle pattern in which content reaches peak performance around 61–90 days and shows a substantial decline in the 271–365 day range. It also reports a rebound for 365+ day content, but explicitly narrows the interpretation: older pages can recover when they are updated well, and the evidence does not show that age naturally reverses performance decline.

**Where does the label/outcome come from?**

The finding is based on content-age buckets and FlyRank health scores across the portfolio. The paper also distinguishes content age from freshness, defined as days since the last update. This distinction is important because an old page can have been recently refreshed.

**My methodology question:**

Does the validation design adequately control for the fact that refreshed older pages may differ systematically from unrefreshed older pages? A stronger test would require comparing comparable groups and ensuring that the refresh occurs before the measured outcome period. Otherwise, the observed association between refresh status and later performance could be affected by selection or survivor bias.

The paper itself identifies survivor bias in the small 365+ × 361+ cell and therefore does not treat that cell as headline evidence. This is a useful example of why sample size and validation design matter when interpreting observed patterns.


In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-09 Section 1: Structured record of paper findings and methodology questions

paper_findings = [
    {
        "finding": "Finding #1 — The Anatomy of Growing Content",
        "outcome_source": "Trend direction based on 30-day vs previous 30-day impression change",
        "methodology_question": (
            "Does the validation design separate descriptive relationships "
            "from predictive evidence, especially when age and word count "
            "are compared with a trend label from the same performance snapshot?"
        )
    },
    {
        "finding": "Finding #2 — The Content Performance Curve",
        "outcome_source": "Content-age buckets compared with FlyRank health scores",
        "methodology_question": (
            "Does the validation design control for differences between "
            "refreshed and unrefreshed older pages and possible survivor or selection bias?"
        )
    }
]

for item in paper_findings:
    print("Finding:", item["finding"])
    print("Outcome source:", item["outcome_source"])
    print("Methodology question:", item["methodology_question"])
    print("-" * 80)

Finding: Finding #1 — The Anatomy of Growing Content
Outcome source: Trend direction based on 30-day vs previous 30-day impression change
Methodology question: Does the validation design separate descriptive relationships from predictive evidence, especially when age and word count are compared with a trend label from the same performance snapshot?
--------------------------------------------------------------------------------
Finding: Finding #2 — The Content Performance Curve
Outcome source: Content-age buckets compared with FlyRank health scores
Methodology question: Does the validation design control for differences between refreshed and unrefreshed older pages and possible survivor or selection bias?
--------------------------------------------------------------------------------


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

## 2. My Model Under an Honest Split — Before vs After

In Week 5, the Logistic Regression model was evaluated using a client-grouped train/test split, with no client overlap between training and testing data.

For this audit, I compare a row-level random split with the grouped-by-client split using the same model features and ranking metric.

* **Before:** random row-level split. This allows rows from the same client to appear in both training and testing data.
* **After:** grouped-by-client split. All rows from a client are kept in only one split, so the test set represents unseen clients.

The comparison is intended to show how the validation design can affect measured Precision@50. The grouped split tests generalization to unseen clients; it does not establish future performance or causality.


In [13]:
# Find the FlyRank dataset location in Colab

import os

possible_paths = [
    "/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv",
    "/content/data/raw/content_refresh_anonymized.csv",
    "/content/content_refresh_anonymized.csv",
]

for path in possible_paths:
    print(path, "->", os.path.exists(path))

print("\nSearching /content for the dataset...\n")

for root, dirs, files in os.walk("/content"):
    if "content_refresh_anonymized.csv" in files:
        print("FOUND:")
        print(os.path.join(root, "content_refresh_anonymized.csv"))

/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv -> True
/content/data/raw/content_refresh_anonymized.csv -> False
/content/content_refresh_anonymized.csv -> False

Searching /content for the dataset...

FOUND:
/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv


In [14]:
from pathlib import Path

print("Searching for content_refresh_anonymized.csv...")

matches = list(Path("/content").rglob("content_refresh_anonymized.csv"))

if matches:
    print("\nFound dataset:")
    for path in matches:
        print(path)
else:
    print("\nDataset was NOT found anywhere under /content.")

Searching for content_refresh_anonymized.csv...

Found dataset:
/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv


In [15]:
from pathlib import Path

repo = Path("/content/flyrank-ml-internship")

print("Repository exists:", repo.exists())

if repo.exists():
    print("\nRepository contents:")
    for item in repo.iterdir():
        print(" -", item)

    data_raw = repo / "data" / "raw"
    print("\nData/raw exists:", data_raw.exists())

    if data_raw.exists():
        print("\nFiles inside data/raw:")
        for item in data_raw.iterdir():
            print(" -", item)

Repository exists: True

Repository contents:
 - /content/flyrank-ml-internship/DATA_USE.md
 - /content/flyrank-ml-internship/GUIDE.md
 - /content/flyrank-ml-internship/outputs
 - /content/flyrank-ml-internship/data
 - /content/flyrank-ml-internship/.gitignore
 - /content/flyrank-ml-internship/skills
 - /content/flyrank-ml-internship/docs
 - /content/flyrank-ml-internship/scripts
 - /content/flyrank-ml-internship/.git
 - /content/flyrank-ml-internship/requirements.txt
 - /content/flyrank-ml-internship/AGENTS.md
 - /content/flyrank-ml-internship/notebooks
 - /content/flyrank-ml-internship/SETUP.md
 - /content/flyrank-ml-internship/README.md
 - /content/flyrank-ml-internship/CLAUDE.md
 - /content/flyrank-ml-internship/LICENSE
 - /content/flyrank-ml-internship/work
 - /content/flyrank-ml-internship/01_first_look_and_discovery.ipynb
 - /content/flyrank-ml-internship/.github
 - /content/flyrank-ml-internship/submission

Data/raw exists: True

Files inside data/raw:
 - /content/flyrank-ml-in

In [16]:
%cd /content

!git clone https://github.com/malikasadnadir-max/flyrank-ml-internship.git

%cd /content/flyrank-ml-internship

/content
fatal: destination path 'flyrank-ml-internship' already exists and is not an empty directory.
/content/flyrank-ml-internship


In [17]:
from pathlib import Path

repo = Path("/content/flyrank-ml-internship")

print("Repository exists:", repo.exists())
print("Data folder exists:", (repo / "data").exists())
print("Raw folder exists:", (repo / "data" / "raw").exists())

print("\nRaw folder contents:")
if (repo / "data" / "raw").exists():
    for item in (repo / "data" / "raw").iterdir():
        print(item)

Repository exists: True
Data folder exists: True
Raw folder exists: True

Raw folder contents:
/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv


In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-09 Section 2: Compare random row split vs grouped-by-client split

import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression


# ---------------------------------------------------------
# 1. Load the Week-5 dataset
# ---------------------------------------------------------

repo_path = Path("/content/flyrank-ml-internship")
data_path = repo_path / "data/raw/content_refresh_anonymized.csv"

if not data_path.exists():
    raise FileNotFoundError(
        f"Dataset not found at: {data_path}\n"
        "Make sure the FlyRank repository is available in Colab."
    )

df = pd.read_csv(data_path)

print("Dataset shape:", df.shape)


# ---------------------------------------------------------
# 2. Define the Week-5 target and model features
# ---------------------------------------------------------

target_col = "is_declining_label"

df[target_col] = (
    df["trend_direction"]
    .astype(str)
    .str.lower()
    .eq("down")
    .astype(int)
)

model_features = [
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "days_since_last_update",
    "content_age_days",
    "word_count"
]

missing_features = [
    col for col in model_features
    if col not in df.columns
]

if missing_features:
    raise ValueError(
        f"Missing model features: {missing_features}"
    )

X = df[model_features].copy()
y = df[target_col].copy()
groups = df["client_id"].copy()


# ---------------------------------------------------------
# 3. Precision@K function
# ---------------------------------------------------------

def precision_at_k(y_true, scores, k=50):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    order = np.argsort(-scores, kind="mergesort")[:k]

    return float(y_true[order].mean())


# ---------------------------------------------------------
# 4. Recreate the Week-4 baseline
# ---------------------------------------------------------

baseline_stale = (
    (df["days_since_last_update"] >= 180)
    & (df["impressions_90d"] >= 500)
).astype(int)

baseline_low_ctr = (
    (df["impressions_90d"] >= 500)
    & (df["avg_position"] > 0)
    & (df["avg_position"] <= 20)
    & (df["ctr"] < 0.5)
).astype(int)

baseline_score = (
    2 * baseline_stale
    + baseline_low_ctr
)


# ---------------------------------------------------------
# 5. Create the Logistic Regression pipeline
# ---------------------------------------------------------

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("logistic_regression", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])


# =========================================================
# BEFORE: Random row-level split
# =========================================================

X_train_random, X_test_random, y_train_random, y_test_random, idx_train_random, idx_test_random = train_test_split(
    X,
    y,
    df.index,
    test_size=0.20,
    random_state=42,
    stratify=y
)

model.fit(X_train_random, y_train_random)

random_scores = model.predict_proba(X_test_random)[:, 1]

random_model_p50 = precision_at_k(
    y_test_random,
    random_scores,
    k=50
)

random_baseline_p50 = precision_at_k(
    y_test_random,
    baseline_score.loc[idx_test_random],
    k=50
)


# Check client overlap in the random split
random_train_clients = set(
    df.loc[idx_train_random, "client_id"]
)

random_test_clients = set(
    df.loc[idx_test_random, "client_id"]
)

random_client_overlap = (
    random_train_clients & random_test_clients
)


# =========================================================
# AFTER: Grouped-by-client split
# =========================================================

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_positions, test_positions = next(
    gss.split(X, y, groups=groups)
)

X_train_grouped = X.iloc[train_positions]
X_test_grouped = X.iloc[test_positions]

y_train_grouped = y.iloc[train_positions]
y_test_grouped = y.iloc[test_positions]

idx_train_grouped = X_train_grouped.index
idx_test_grouped = X_test_grouped.index

model.fit(X_train_grouped, y_train_grouped)

grouped_scores = model.predict_proba(
    X_test_grouped
)[:, 1]

grouped_model_p50 = precision_at_k(
    y_test_grouped,
    grouped_scores,
    k=50
)

grouped_baseline_p50 = precision_at_k(
    y_test_grouped,
    baseline_score.loc[idx_test_grouped],
    k=50
)


# Check client overlap in grouped split
grouped_train_clients = set(
    df.loc[idx_train_grouped, "client_id"]
)

grouped_test_clients = set(
    df.loc[idx_test_grouped, "client_id"]
)

grouped_client_overlap = (
    grouped_train_clients & grouped_test_clients
)


# ---------------------------------------------------------
# 6. Display before vs after results
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("ML-09 VALIDATION COMPARISON")
print("=" * 70)

print("\nBEFORE — Random row-level split")
print("Train rows:", len(idx_train_random))
print("Test rows:", len(idx_test_random))
print("Train clients:", len(random_train_clients))
print("Test clients:", len(random_test_clients))
print("Client overlap:", len(random_client_overlap))
print(f"Baseline Precision@50: {random_baseline_p50:.3f}")
print(f"Model Precision@50:    {random_model_p50:.3f}")

print("\nAFTER — Grouped-by-client split")
print("Train rows:", len(idx_train_grouped))
print("Test rows:", len(idx_test_grouped))
print("Train clients:", len(grouped_train_clients))
print("Test clients:", len(grouped_test_clients))
print("Client overlap:", len(grouped_client_overlap))
print(f"Baseline Precision@50: {grouped_baseline_p50:.3f}")
print(f"Model Precision@50:    {grouped_model_p50:.3f}")

print("\n" + "-" * 70)
print("Change in measured model Precision@50:")
print(
    f"{grouped_model_p50 - random_model_p50:+.3f}"
)
print("-" * 70)


# ---------------------------------------------------------
# 7. Basic validation checks
# ---------------------------------------------------------

assert len(grouped_client_overlap) == 0, (
    "Grouped split contains client overlap."
)

assert len(idx_train_grouped) > 0
assert len(idx_test_grouped) > 0

assert not set(model_features) & {
    "is_declining_label",
    "trend_direction",
    "trend_pct",
    "future_clicks",
    "future_impressions",
    "future_sessions",
    "next_30d_clicks",
    "next_30d_impressions",
    "next_30d_sessions"
}

print("\nSection 2 checks: PASS")

Dataset shape: (30000, 44)

ML-09 VALIDATION COMPARISON

BEFORE — Random row-level split
Train rows: 24000
Test rows: 6000
Train clients: 32
Test clients: 31
Client overlap: 31
Baseline Precision@50: 0.720
Model Precision@50:    0.720

AFTER — Grouped-by-client split
Train rows: 23837
Test rows: 6163
Train clients: 25
Test clients: 7
Client overlap: 0
Baseline Precision@50: 0.540
Model Precision@50:    0.600

----------------------------------------------------------------------
Change in measured model Precision@50:
-0.120
----------------------------------------------------------------------

Section 2 checks: PASS


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## 3. Leakage Audit

I audited the final feature set used by the Week-5 Logistic Regression model:

* `impressions_90d`
* `clicks_90d`
* `ctr`
* `avg_position`
* `days_since_last_update`
* `content_age_days`
* `word_count`

The audit identified `trend_direction` and `trend_pct` as potentially suspicious columns in the dataset because they are directly related to the trend outcome. However, neither of these columns is included in the final feature set.

The audit also checked for known target or future-performance fields, including the target label and future or next-30-day outcome fields. None of these fields is included in the final feature set.

Therefore, based on this feature-level audit, I found no direct target or future-outcome leakage in the final feature set. This supports treating the selected features as leakage-safe for this audit, while recognizing that feature-level checks alone do not prove the absence of every possible source of leakage.


In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-09 Section 3: Leakage audit

import pandas as pd
from pathlib import Path

# ---------------------------------------------------------
# 1. Load the dataset
# ---------------------------------------------------------

repo_path = Path("/content/flyrank-ml-internship")
data_path = repo_path / "data/raw/content_refresh_anonymized.csv"

df_leakage = pd.read_csv(data_path)

# ---------------------------------------------------------
# 2. Final feature set from Week 5
# ---------------------------------------------------------

final_features = [
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "days_since_last_update",
    "content_age_days",
    "word_count"
]

target_col = "is_declining_label"

# ---------------------------------------------------------
# 3. Check that all final features exist
# ---------------------------------------------------------

missing_features = [
    col for col in final_features
    if col not in df_leakage.columns
]

assert not missing_features, (
    f"Missing final features: {missing_features}"
)

print("Final feature set:")
for feature in final_features:
    print(" -", feature)

# ---------------------------------------------------------
# 4. Look for suspicious target/future columns
# ---------------------------------------------------------

suspicious_keywords = [
    "future",
    "next_30d",
    "trend_pct",
    "trend_direction",
    "label",
    "target"
]

suspicious_columns = [
    col for col in df_leakage.columns
    if any(keyword in col.lower() for keyword in suspicious_keywords)
]

print("\nPotentially suspicious columns:")
for col in suspicious_columns:
    print(" -", col)

# ---------------------------------------------------------
# 5. Check whether suspicious columns are in final features
# ---------------------------------------------------------

leakage_candidates = [
    col for col in final_features
    if any(keyword in col.lower() for keyword in suspicious_keywords)
]

print("\nSuspicious columns included in final feature set:")
print(leakage_candidates)

# ---------------------------------------------------------
# 6. Explicitly check known future/target fields
# ---------------------------------------------------------

known_leakage_fields = {
    "trend_direction",
    "trend_pct",
    "future_clicks",
    "future_impressions",
    "future_sessions",
    "next_30d_clicks",
    "next_30d_impressions",
    "next_30d_sessions",
    "is_declining_label"
}

used_known_leakage = [
    col for col in final_features
    if col in known_leakage_fields
]

print("\nKnown target/future fields used:")
print(used_known_leakage)

# ---------------------------------------------------------
# 7. Basic audit checks
# ---------------------------------------------------------

assert target_col not in final_features
assert len(leakage_candidates) == 0
assert len(used_known_leakage) == 0

print("\nSection 3 checks: PASS")

Final feature set:
 - impressions_90d
 - clicks_90d
 - ctr
 - avg_position
 - days_since_last_update
 - content_age_days
 - word_count

Potentially suspicious columns:
 - trend_direction
 - trend_pct

Suspicious columns included in final feature set:
[]

Known target/future fields used:
[]

Section 3 checks: PASS


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## 4. Claim Rewrite

### Original claim

The Logistic Regression model can accurately identify declining content and outperform the baseline.

### Safer claim

Under the grouped-by-client validation used in this audit, the Logistic Regression model achieved a measured Precision@50 of 0.600, compared with 0.540 for the baseline. This is an observed directional improvement of 0.060 in Precision@50 on the held-out client group. The result supports using the model as decision-support for prioritizing potentially declining pages, but it does not establish causal effects or guarantee performance on future clients or time periods.


In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-09 Section 4: Claim rewrite — numerical check

model_p50 = 0.600
baseline_p50 = 0.540

difference = model_p50 - baseline_p50

print("Model Precision@50:", model_p50)
print("Baseline Precision@50:", baseline_p50)
print("Observed difference:", f"{difference:+.3f}")

# Round before checking because of floating-point precision
assert round(model_p50, 3) == 0.600
assert round(baseline_p50, 3) == 0.540
assert round(difference, 3) == 0.060

print("\nSection 4 checks: PASS")

Model Precision@50: 0.6
Baseline Precision@50: 0.54
Observed difference: +0.060

Section 4 checks: PASS


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.